# 🐖 15 — Conteo de cerdos: corrección de umbrales (CPU)

**Propósito:** reparar la comparación sesgada por utilizar 0,25 en ambos modelos. Se selecciona el umbral en validación y SOLO después se evalúa en test. Se preservan 00–14 y los pesos originales. **Dataset público FaroPigSeg; nunca cerdos únicos**. Fórmula de MAE y búsqueda verificadas con los ZIP 09 y 13.

**Criterio congelado:** confianza 0,01 a 0,99 (paso 0,01), minimizar MAE de conteo en 196 imágenes de validación, desempate por menor umbral. Se informa también el umbral común 0,25 sin reemplazar los resultados originales.

## 1. Dependencias (CPU)

In [ ]:
%pip -q install "pandas>=2,<3" numpy "openpyxl>=3.1" matplotlib

## 2. Archivos históricos y controles de integridad

In [ ]:
from google.colab import drive,files
from pathlib import Path
from collections import Counter,defaultdict
from datetime import datetime,timezone
import json,zipfile,hashlib,math
import numpy as np,pandas as pd,matplotlib.pyplot as plt
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'No encuentro {ROOT}; revisa la cuenta de Google'
REPORT=ROOT/'03_Resultados'/'05_Reportes'; METRIC=ROOT/'03_Resultados'/'03_Metricas'
FIG=ROOT/'03_Resultados'/'04_Figuras'; EXPORT=ROOT/'05_Exportaciones_ZIP'
for d in (REPORT,METRIC,FIG,EXPORT):d.mkdir(parents=True,exist_ok=True)
D09=REPORT/'CERDOS_09_COCO_Armonizado';D13=REPORT/'CERDOS_13_Calibracion_Validacion'
Y='YOLO11n-seg';M='Mask R-CNN'
def readj(p):
    assert p.is_file(),f'No encuentro: {p}'
    return json.loads(p.read_text(encoding='utf8'))
GT={'val':readj(D13/'CERDOS_13_GT_val_coco.json'),'test':readj(D09/'CERDOS_09_GT_test_coco.json')}
PREDS={
 ('val',Y):readj(D13/'CERDOS_13_Pred_YOLO_val_coco.json'),
 ('val',M):readj(D13/'CERDOS_13_Pred_MaskRCNN_val_coco.json'),
 ('test',Y):readj(D09/'CERDOS_09_Pred_YOLO_coco.json'),
 ('test',M):readj(D09/'CERDOS_09_Pred_MaskRCNN_coco.json')}
assert len(GT['val']['images'])==196 and len(GT['test']['images'])==373
assert len(GT['val']['annotations'])==1995 and len(GT['test']['annotations'])==4648
P09=readj(D09/'CERDOS_09_Protocolo.json');P13=readj(D13/'CERDOS_13_Protocolo.json')
for key in ('yaml_sha256','yolo_sha256','mask_sha256'):
    assert P09[key]==P13[key],f'Protocolos distintos: {key}'
print('✅ Predicciones previas y pesos verificados. No se necesita GPU ni reentrenar.')


## 3. Selección SOLO en validación y evaluación posterior en prueba

In [ ]:
# Cuadrícula y criterio definidos antes de inspeccionar test
GRID=np.round(np.arange(.01,1,.01),2)
BASELINE=.25
rows=[]; curves=[];perimage=[]
for split in ('val','test'):
    gt=GT[split]
    imgs={int(i['id']):i['file_name'] for i in gt['images']}
    true=Counter(int(a['image_id']) for a in gt['annotations'])
    for model in (Y,M):
        scores=defaultdict(list)
        for x in PREDS[split,model]:scores[int(x['image_id'])].append(float(x['score']))
        for t in GRID:
            err=np.asarray([sum(s>=t for s in scores[i])-true.get(i,0) for i in imgs])
            curves.append({'split':split,'modelo':model,'umbral':float(t),
                           'MAE':float(np.abs(err).mean()),'RMSE':float(np.sqrt(np.mean(err**2))),
                           'sesgo_conteo':float(err.mean())})
C=pd.DataFrame(curves)
sel=[]
for model in (Y,M):
    a=C[(C.split=='val')&(C.modelo==model)]
    best=a.sort_values(['MAE','umbral'],ascending=[True,True],kind='stable').iloc[0]
    sel.append({'modelo':model,'umbral_validacion':float(best.umbral),'MAE_val':float(best.MAE),
                'criterio':'min MAE en validación; empates: umbral menor; grid 0.01-0.99 paso 0.01'})
S=pd.DataFrame(sel)
for split in ('val','test'):
    gt=GT[split];imgs={int(i['id']):i['file_name'] for i in gt['images']}
    true=Counter(int(a['image_id']) for a in gt['annotations'])
    for model in (Y,M):
        scores=defaultdict(list)
        for x in PREDS[split,model]:scores[int(x['image_id'])].append(float(x['score']))
        optimal=float(S.loc[S.modelo.eq(model),'umbral_validacion'].iloc[0])
        for tag,t in [('Fijo 0.25',BASELINE),('Elegido en val',optimal)]:
            for i,filename in imgs.items():
                pred=sum(s>=t for s in scores[i]);n_gt=true.get(i,0)
                perimage.append({'split':split,'modelo':model,'escenario':tag,'umbral':float(t),
                                 'imagen_id':i,'imagen':filename,'fecha':filename[:10],
                                 'familia':'all_in_one' if 'all_in_one' in filename else 'left_cropped' if 'left_cropped' in filename else 'otra',
                                 'GT':n_gt,'predichos':pred,'error':pred-n_gt,'error_abs':abs(pred-n_gt)})
P=pd.DataFrame(perimage)
summary=P.groupby(['split','modelo','escenario','umbral'],as_index=False).agg(
    imagenes=('imagen','size'),MAE=('error_abs','mean'),sesgo=('error','mean'))
summary['RMSE']=[float(np.sqrt(np.mean(g.error.to_numpy()**2))) for _,g in P.groupby(['split','modelo','escenario','umbral'],sort=True)]
print('✅ Umbrales elegidos exclusivamente en validación');display(S)
print('✅ Prueba, solo evaluación de los umbrales ya congelados');display(summary[summary.split.eq('test')])


## 4. Excel, figura y ZIP en Drive

In [ ]:
OUT=METRIC/'CERDOS_15_Correccion_Conteo_Val.xlsx'
FIGURE=FIG/'CERDOS_15_Sensibilidad_Umbral_Conteo.png'
J=REPORT/'CERDOS_15_Conteo_Protocolo.json'
with pd.ExcelWriter(OUT,engine='openpyxl') as w:
    S.to_excel(w,sheet_name='Umbrales_val',index=False)
    summary.to_excel(w,sheet_name='Resumen',index=False)
    C.to_excel(w,sheet_name='Curvas_umbral',index=False)
    P.to_excel(w,sheet_name='Por_imagen',index=False)
fig,ax=plt.subplots(figsize=(10,5.5))
color={Y:'#28618A',M:'#D37F58'}
for model in (Y,M):
    x=C[(C.modelo==model)&(C.split=='val')]
    ax.plot(x.umbral,x.MAE,linestyle='-',color=color[model],lw=1.7,
            label=f'{model} · validation')
    threshold=float(S.loc[S.modelo.eq(model),'umbral_validacion'].iloc[0])
    ax.axvline(threshold,color=color[model],lw=1,alpha=.55,ls=':')
ax.set(xlabel='Confidence threshold',ylabel='Counting MAE (pigs per image)',title='Pig counting: validation-selected confidence thresholds')
ax.grid(alpha=.15);ax.legend(ncol=2,fontsize=9);fig.tight_layout();fig.savefig(FIGURE,dpi=400);plt.close(fig)
meta={'fase':'15','fecha_utc':datetime.now(timezone.utc).isoformat(),
      'protocolo':'conteo por imagen, detecciones score >= umbral; sin nueva inferencia',
      'seleccion':'umbral fijo 0.25 + grid 0.01..0.99; minimizar MAE exclusivamente en val; empates umbral menor',
      'umbral_configurado_en_val':S.to_dict('records'),
      'resultados_test':summary[summary.split.eq('test')].to_dict('records'),
      'limitaciones':['El conteo por imagen no identifica cerdos físicos únicos.',
                     'Test reutilizado de análisis previos; confirmatorio/exploratorio, no externo.',
                     'Los intervalos/efectos por imagen no presuponen independencia dentro de fechas/sesiones.',
                     'Las conclusiones dependen del postprocesamiento, NMS y umbrales.']}
J.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf8')
ZIP=EXPORT/'CERDOS_15_Correccion_Conteo_Resultados.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in (OUT,FIGURE,J):z.write(p,arcname=p.name)
assert zipfile.ZipFile(ZIP).testzip() is None
print('✅ Archivos guardados:',OUT,FIGURE,ZIP)
files.download(str(ZIP))
